# Piloto de fine-tuning — español sintético con XTTS v2 base

XTTS v2 base es el profesor con mejor acento: 4,75 de acento nativo en la escucha a ciegas. Alucina en bloques largos, pero aquí cada clip es una sola frase y el cuaderno 03 descarta los que fallen. Parámetros de tu cuaderno de producción: `temperature=0.75`, `repetition_penalty=3.0`, `top_p=0.8`, `top_k=60`.

Genera las **400 frases** de `piloto/frases_sinteticas.json` (unos 45 min de audio) con la voz de Naval,
usando `speaker_refs/naval/naval_ref02.wav` como referencia. Salida: `fine_tunned_naval/piloto/sintetico/`
en Drive, un `.wav` por frase (`s001` … `s400`).

**GPU:** T4 (o L4 si no hay T4). **Tiempo estimado:** unos 40 min.

Uso: celda 1, *Entorno de ejecución → Reiniciar sesión*, y el resto en orden. Si se corta, vuelve a empezar:
lo que ya esté en Drive no se genera otra vez. La última celda sube todo a Drive y desconecta la GPU.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q coqui-tts==0.27.5 "coqui-tts-trainer>=0.3.0,<0.4.0" "transformers==4.57.1"

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencia de Naval
import os, sys, subprocess
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun

comun.montar_drive()
comun.FRASES = Path("/content/repo/piloto/frases_sinteticas.json")
comun.SALIDA = comun.DRIVE / "piloto" / "sintetico"
ITEMS, SILENCIO_MS = comun.cargar_items()
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Generar
import os
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS
VARIANTE = "xtts_base"
GEN = dict(temperature=0.75, repetition_penalty=3.0, top_p=0.8, top_k=60, length_penalty=1.0,
           enable_text_splitting=False)
modelo = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda").synthesizer.tts_model
GPT, SPK = modelo.get_conditioning_latents(audio_path=[REFS["naval"]["wav"]], gpt_cond_len=12,
                                           gpt_cond_chunk_len=6, max_ref_length=12)

def sintetizar(texto, ref):
    return modelo.inference(texto, "es", GPT, SPK, **GEN)["wav"], 24000

comun.guardar_entorno(VARIANTE, ["coqui-tts", "torch", "transformers"])
comun.generar(VARIANTE, sintetizar, ITEMS, REFS, SILENCIO_MS, params=GEN)

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
filas = [json.loads(l) for l in (comun.SALIDA / "xtts_base" / "registro.jsonl").read_text(encoding="utf-8").splitlines()]
ok = sum(f["ok"] for f in filas)
minutos = sum(f.get("segundos_audio", 0) for f in filas if f["ok"]) / 60
print(f"xtts_base: {ok} clips correctos ({minutos:.0f} min), {len(filas) - ok} con error")
comun.desconectar()